# OneClickAI 시계열 예측
이 노트북은 Google Colab 환경에서 실행하도록 작성되었습니다.

**순서**
1. 파일 업로드 및 데이터 로딩
2. 데이터 전처리 및 모델 학습
3. 결과 확인 및 미래 예측

## 1. 파일 업로드 및 데이터 로딩

엑셀 파일(`.xlsx`, `.xls`)을 업로드해 주세요.  
파일에는 **원인** 시트와 **결과** 시트가 필요합니다.

In [ ]:
# ==========================================
# 파일 업로드
# ==========================================
from google.colab import files
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Input
import matplotlib.pyplot as plt

print("데이터 파일(.xlsx, .xls)을 업로드 해주세요.\n")
uploaded = files.upload()

if len(uploaded.keys()) == 0:
    raise ValueError("파일이 업로드되지 않았습니다.")

DATA_FILE_PATH = list(uploaded.keys())[0]

if not DATA_FILE_PATH.endswith(('.xls', '.xlsx')):
    raise ValueError("지원하지 않는 파일 형식입니다. .xlsx 또는 .xls 파일을 업로드 해주세요.")


# ==========================================
# 파일 열기
# ==========================================
df_x = pd.read_excel(DATA_FILE_PATH, sheet_name='원인', index_col=0)
df_y = pd.read_excel(DATA_FILE_PATH, sheet_name='결과', index_col=0)
print(f"데이터 로딩 완료: {DATA_FILE_PATH} (원인: {df_x.shape}, 결과: {df_y.shape})\n")


## 2. 데이터 전처리 및 모델 학습

데이터를 정규화하고 시퀀스로 변환한 뒤 LSTM 모델을 학습합니다.

- `SEQ_LENGTH`: 과거 몇 개의 데이터를 입력으로 사용할지 설정
- `PRED_LENGTH`: 미래 몇 개의 데이터를 예측할지 설정

In [ ]:
# ==========================================
# 데이터 확인
# ==========================================
varname_x = list(df_x.columns)
varname_y = list(df_y.columns)
len_x, len_y, len_data = len(df_x.columns), len(df_y.columns), len(df_x)
print(f'독립변수: {varname_x}\n')
print(f'종속변수: {varname_y}\n')
print(f'독립변수 {len_x}개, 종속변수 {len_y}개, 데이터 {len_data}개\n')

# ==========================================
# 데이터 전처리 (Min-Max 정규화)
# ==========================================
x = np.array(df_x).astype(float)
y = np.array(df_y).astype(float)
x = (x - x.min()) / (x.max() - x.min())
y_min, y_max = y.min(), y.max()
y = (y - y_min) / (y_max - y_min)

# ==========================================
# 시퀀스 데이터 생성
# ==========================================
SEQ_LENGTH  =  5  # 과거 몇 개의 데이터를 입력으로 사용할지 설정
PRED_LENGTH =  5  # 미래 몇 개의 데이터를 예측할지 설정

x_seq, y_seq = [], []
for i in range(len_data - SEQ_LENGTH - PRED_LENGTH + 1):
    x_seq.append(x[i:i + SEQ_LENGTH])
    y_seq.append(y[i + SEQ_LENGTH:i + SEQ_LENGTH + PRED_LENGTH].flatten())

x_seq = np.array(x_seq)  # (샘플 수, SEQ_LENGTH, 독립변수 수)
y_seq = np.array(y_seq)  # (샘플 수, PRED_LENGTH * 종속변수 수)
print(f'시퀀스 데이터 생성 완료: x={x_seq.shape}, y={y_seq.shape}\n')

# ==========================================
# 데이터를 training set과 validation set으로 나누기
# ==========================================
split_idx = int(len(x_seq) * 0.8)
x_train, x_val = x_seq[:split_idx], x_seq[split_idx:]
y_train, y_val = y_seq[:split_idx], y_seq[split_idx:]
print(f'training: {len(x_train)}개, validation: {len(x_val)}개\n')

# ==========================================
# 모델 생성
# ==========================================
model = tf.keras.Sequential()
model.add(Input(shape=(SEQ_LENGTH, len_x)))
model.add(LSTM(64, activation='tanh'))
model.add(Dense(32, activation='relu'))
model.add(Dense(PRED_LENGTH * len_y))
model.summary()

# ==========================================
# 모델 설정 및 학습
# ==========================================
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), batch_size=16, epochs=100)

# ==========================================
# 모델 저장 및 불러오기
# ==========================================
model.save('timeseries_model.keras')
model_loaded = tf.keras.models.load_model('timeseries_model.keras')


## 3. 결과 확인 및 미래 예측

학습 과정의 손실(MSE) / MAE 그래프를 출력하고, 마지막 시퀀스를 사용해 미래 값을 예측합니다.

In [ ]:
# ==========================================
# 모델 결과 확인 (그래프)
# ==========================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper right')

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'])
plt.plot(history.history['val_mae'])
plt.title('model MAE')
plt.ylabel('MAE')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper right')

plt.show()

# ==========================================
# 테스트 및 시각화
# ==========================================
# 전체 실제 데이터 (역변환)
full_actual = y * (y_max - y_min) + y_min  # (len_data, len_y)

# 마지막 SEQ_LENGTH개로 미래 PRED_LENGTH개 예측
last_seq = x[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, len_x)
future_pred = model.predict(last_seq).reshape(PRED_LENGTH, len_y) * (y_max - y_min) + y_min

# 전체 데이터 + 미래 예측 시각화 (마지막 실제값에서 예측값으로 이어지도록)
connect = np.concatenate([[full_actual[-1, 0]], future_pred[:, 0]])

plt.figure(figsize=(15, 5))
plt.plot(range(len_data), full_actual[:, 0], label='Actual', marker='x')
plt.plot(range(len_data - 1, len_data + PRED_LENGTH), connect, label='Predicted', linestyle='--', marker='o')
plt.axvline(x=len_data - 1, color='gray', linestyle=':', label='Prediction Start')
plt.title('Full Data + Future Prediction')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

# 미래 예측값 출력
print("미래 예측값:")
for step in range(PRED_LENGTH):
    for name, val in zip(varname_y, future_pred[step]):
        print(f"  Step {step+1} - {name}: {val:.4f}\n")